# Moshi Compression — Session S1 (Phase 0 teacher caching + frozen-heads export)

**Goal.** Produce two persisted artifacts so every later session can train without
loading the 7.7 B teacher:

1. `moshi-teacher-cache-pilot` — LMDB of per-window `(h_T_final fp16, top-256 text
   logits, semantic cb0 logits)` for ~600 × 30 s windows of LibriLight.
2. `moshi-frozen-heads` — state_dicts for student-side frozen modules
   (`emb`, `text_emb`, `out_norm`, `text_linear`, `depformer_in`, `depformer`,
   `depformer_emb`, `depformer_text_emb`, `linears`), copied from the teacher.

**Why frozen heads matter.** S0 built the student shell with `load_weight=False`,
so its shared modules are random-init. Phase 1 training needs them initialised
from the teacher, otherwise the KD signal is meaningless at step 0.

**What carries over from S0** (see `MoshiCompressionProgress.md` §S0 findings):
* Patches: `torch.compile` disabled pre-import, `CUDAGraphed` no-op post-install.
* Teacher sharded layers 0–15 on cuda:0, 16–31 + heads on cuda:1.
* `_sharded_forward` monkey-patch for cross-GPU routing.
* Mimi on cuda:0 fp16.

**What is NEW in S1**:
* LMDB cache writer with a fixed key schema.
* Mimi encoding of 30 s @ 24 kHz windows → 375 frames × 8 audio codebooks.
* Teacher `forward_text` in `torch.inference_mode()` — no student, no backward.
* Throughput measurement (`S1_throughput.json`).
* Two Kaggle Dataset pushes at the end.

**Exit criteria** (must all be ✅ before calling S1 done):
* `teacher_cache_pilot/*.lmdb` ≥ 500 windows on disk, manifest lists keys.
* `moshi-frozen-heads` dataset uploaded with every module above.
* `S1_throughput.json` contains `windows_per_minute` (per-GPU-hour extrapolation
  decides how many S2..Sk sessions the full 60 k cache will take).


## Cell 1 — Global patches

`torch.compile` must be disabled **before** any `moshi` import; T4 Inductor
still emits bf16 intrinsics that crash with “no kernel image available”.


In [1]:
# Disable torch.compile globally.
# Kaggle T4 Inductor occasionally emits bf16 intrinsics → "no kernel image" crash.
import os
import torch

os.environ["TORCH_COMPILE_DISABLE"] = "1"
torch._dynamo.config.disable = True
print("torch.compile disabled globally")
print("NOTE: CUDAGraphed patch will be applied in Cell 3b, after moshi is installed.")


torch.compile disabled globally
NOTE: CUDAGraphed patch will be applied in Cell 3b, after moshi is installed.


## Cell 2 — Environment verification


In [2]:
import sys
print("python :", sys.version)
print("torch  :", torch.__version__, "  cuda:", torch.version.cuda)
print("cuda available  :", torch.cuda.is_available())
print("device count    :", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, sm {p.major}.{p.minor}, "
          f"total {p.total_memory / 1e9:.1f} GB")

# ── Hard assertions ──
assert torch.cuda.device_count() == 2, (
    f"Need dual-GPU Kaggle runtime, got {torch.cuda.device_count()} GPU(s)")

for i in range(2):
    p = torch.cuda.get_device_properties(i)
    assert (p.major, p.minor) == (7, 5), (
        f"Expected T4 (sm_75) on cuda:{i}, got sm_{p.major}.{p.minor}")
    assert p.total_memory >= 15_000_000_000, (
        f"cuda:{i} reports only {p.total_memory/1e9:.1f} GB, expected ~16 GB")

def hw_bf16_supported():
    return all(
        torch.cuda.get_device_capability(i)[0] >= 8
        for i in range(torch.cuda.device_count())
    )

print("bf16 supported (hardware CC>=8.0):", hw_bf16_supported())
assert not hw_bf16_supported(), (
    "Hardware bf16 detected (CC >= 8.0) — are you on a non-T4 GPU? "
    f"Capabilities: {[torch.cuda.get_device_capability(i) for i in range(torch.cuda.device_count())]}"
)

try:
    import flash_attn  # noqa
    print("WARNING: flash_attn imported — unexpected on T4")
except ImportError:
    print("flash_attn not present — expected on T4")

# Anchor cuda:0 so Kaggle GPU ordering stays stable.
torch.cuda.set_device(0)
print("cuda:0 anchored")
print("=== environment check PASSED ===")


python : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
torch  : 2.10.0+cu128   cuda: 12.8
cuda available  : True
device count    : 2
  cuda:0 = Tesla T4, sm 7.5, total 15.6 GB
  cuda:1 = Tesla T4, sm 7.5, total 15.6 GB
bf16 supported (hardware CC>=8.0): False
flash_attn not present — expected on T4
cuda:0 anchored
=== environment check PASSED ===


## Cell 3 — Pinned installs

Same pins as S0 plus `lmdb` (cache writer) and `soundfile`/`librosa` (audio
resampling to Mimi's 24 kHz).


In [3]:
import subprocess, sys, os

# PyPI packages — pinned versions, non-negotiable.
for pkg in [
    "transformers==4.44.2",
    "numpy>=1.24,<2.0",            # pin to avoid numpy.char breakage
    "soxr",                        # fast resampler, no scipy dep
    "accelerate==0.33.0",
    "bitsandbytes>=0.45,<0.50",  # moshi 0.2.13 requires >=0.45; 0.49.x is fine
    "sentencepiece",
    "einops",
    "lmdb",                      # NEW in S1: teacher-target cache backing store
    "soundfile",                 # NEW in S1: 16-bit WAV reader
    "datasets",                  # HF streaming audio source
]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# ── Moshi — copy to /kaggle/working first, then install ──────────────────────
MOSHI_SRC = "/kaggle/input/datasets/tasfiatanha/moshi-repo/moshi/moshi"
MOSHI_DST = "/kaggle/working/moshi_repo"

if not os.path.exists(MOSHI_DST):
    print(f"Copying moshi repo to {MOSHI_DST} …")
    ret = subprocess.run(["cp", "-r", MOSHI_SRC, MOSHI_DST], capture_output=True, text=True)
    if ret.returncode != 0:
        raise RuntimeError(f"cp failed:\n{ret.stderr}")
    print("Copy done")
else:
    print(f"Repo already at {MOSHI_DST}, skipping copy")

print("Installing moshi (editable) …")
ret = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", MOSHI_DST],
    capture_output=True, text=True
)
if ret.returncode != 0:
    print("pip stdout:", ret.stdout)
    print("pip stderr:", ret.stderr)
    raise RuntimeError("moshi editable install failed — see output above")
print("moshi installed (editable)")

import site, importlib
site.addsitedir(site.getsitepackages()[0])
MOSHI_SRC_ROOT = MOSHI_DST
if MOSHI_SRC_ROOT not in sys.path:
    sys.path.insert(0, MOSHI_SRC_ROOT)
importlib.invalidate_caches()

try:
    import moshi
    print(f"moshi importable from: {moshi.__file__}")
except ModuleNotFoundError as e:
    raise RuntimeError(
        f"moshi still not importable after sys.path fix: {e}\n"
        f"sys.path = {sys.path}"
    )

print("transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("lmdb         :", lmdb.__version__)
print("soundfile    :", soundfile.__version__)
print("=== installs OK ===")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 100.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 73.2 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.27.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 315.1/315.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.1/333.1 kB 13.2 MB/s eta 0:00:00
Copying moshi repo to /kaggle/working/moshi_repo …
Copy done
Installing moshi (editable) …
moshi installed (editable)
moshi importable from: /kaggle/working/moshi_repo/moshi/__init__.py


NameError: name 'transformers' is not defined

In [4]:
import transformers, bitsandbytes as bnb, lmdb, soundfile, soxr
print("transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("lmdb        :", lmdb.__version__)
print("soundfile   :", soundfile.__version__)
print("soxr        :", soxr.__version__)

transformers: 4.44.2
bitsandbytes: 0.49.2
lmdb        : 2.2.0
soundfile   : 0.13.1
soxr        : 1.0.0


In [5]:
# Cell 3b — Monkey-patch CUDAGraphed to a no-op.
# Must run AFTER moshi is installed (Cell 3) but BEFORE any import of
# moshi.models.lm_gen. In S1 we only call `forward_text` inside
# `torch.inference_mode()`, but the no-op is still cheap insurance in case the
# teacher path grows a CUDAGraphed wrapper later.

import moshi.utils.compile as _moshi_compile

class _NoGraph:
    def __init__(self, fn, *a, **kw): self.fn = fn
    def __call__(self, *a, **kw):    return self.fn(*a, **kw)

_moshi_compile.CUDAGraphed = _NoGraph
print("CUDAGraphed monkey-patched to no-op")

import torch
assert torch._dynamo.config.disable, "torch.compile should still be disabled"
print("torch.compile still disabled — both patches active")


CUDAGraphed monkey-patched to no-op
torch.compile still disabled — both patches active


## Cell 4 — Load teacher + Mimi (fp16) and shard across GPUs

Identical to S0 Cell 4: download weights to `/tmp`, construct `CheckpointInfo`
directly (no `from_pretrained`), load teacher+Mimi on CPU, shard layers 0–15 to
cuda:0 and 16–31 + heads to cuda:1, install `_sharded_forward`. Teacher stays
in `eval()` mode for S1 — we never backprop through it in this session.


In [6]:
import torch, pathlib, time, shutil
from moshi.models.loaders import CheckpointInfo

REPO_ID  = "kyutai/moshiko-pytorch-bf16"
LOAD_DIR = pathlib.Path("/tmp/moshiko-weights")
LOAD_DIR.mkdir(parents=True, exist_ok=True)

MOSHI_FILE = "model.safetensors"
MIMI_FILE  = "tokenizer-e351c8d8-checkpoint125.safetensors"
TOK_FILE   = "tokenizer_spm_32k_3.model"
FILES = {
    MOSHI_FILE : 14_000_000_000,
    MIMI_FILE  : 350_000_000,
    TOK_FILE   : 500_000,
}

def already_done(filename, min_size):
    p = LOAD_DIR / filename
    return p.exists() and p.stat().st_size >= min_size

from huggingface_hub import hf_hub_download

all_ok = all(already_done(f, s) for f, s in FILES.items())
if all_ok:
    print("All files already in /tmp — skipping download")
else:
    for filename, min_size in FILES.items():
        if already_done(filename, min_size):
            print(f"SKIP {filename} (already complete)")
            continue
        attempt = 0
        while True:
            attempt += 1
            current = (LOAD_DIR / filename).stat().st_size if (LOAD_DIR / filename).exists() else 0
            print(f"[attempt {attempt}] {filename}  {current/1e9:.2f} / ~{min_size/1e9:.2f} GB")
            try:
                hf_hub_download(
                    repo_id   = REPO_ID,
                    filename  = filename,
                    local_dir = str(LOAD_DIR),
                    force_download = False,
                )
                if already_done(filename, min_size):
                    print("  OK")
                    break
            except Exception as e:
                print(f"  Error: {e} — retrying in 10s")
                time.sleep(10)

moshi_weights = LOAD_DIR / MOSHI_FILE
mimi_weights  = LOAD_DIR / MIMI_FILE
tokenizer     = LOAD_DIR / TOK_FILE
for f in [moshi_weights, mimi_weights, tokenizer]:
    assert f.exists(), f"Missing: {f.name}"
    print(f"  {f.name:55s} {f.stat().st_size/1e9:.3f} GB")

info = CheckpointInfo(
    moshi_weights = moshi_weights,
    mimi_weights  = mimi_weights,
    tokenizer     = tokenizer,
    lm_config     = None,
)

print("\nLoading teacher LM to CPU …")
teacher_lm = info.get_moshi(device="cpu", dtype=torch.float16)
print("Loading Mimi to CPU …")
mimi = info.get_mimi(device="cpu")

def count_params(m):
    return sum(p.numel() for p in m.parameters())

print(f"\nTeacher LM params     : {count_params(teacher_lm) / 1e9:.3f} B")
print(f"Teacher LM mem (fp16) : {count_params(teacher_lm) * 2 / 1e9:.2f} GB")
print(f"Mimi params           : {count_params(mimi) / 1e6:.1f} M")

print("\nDeleting weights from /tmp (loaded into RAM, disk copy not needed) …")
shutil.rmtree(LOAD_DIR)
print(f"Deleted {LOAD_DIR}")

stat = __import__('os').statvfs("/kaggle/working")
print(f"Disk free: {stat.f_bavail * stat.f_frsize / 1e9:.1f} GB")

print("\nClearing GPU memory before shard …")
torch.cuda.empty_cache()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  cuda:{i} before shard: free {free/1e9:.2f} / total {total/1e9:.2f} GB")

print("\nSharding teacher layer-by-layer …")
teacher_lm.emb.to("cuda:0")
teacher_lm.text_emb.to("cuda:0")

SPLIT = 16
for layer in teacher_lm.transformer.layers[:SPLIT]:
    layer.to("cuda:0")
print(f"  Layers  0-{SPLIT-1} → cuda:0")

for layer in teacher_lm.transformer.layers[SPLIT:]:
    layer.to("cuda:1")
print(f"  Layers {SPLIT}-31 → cuda:1")

for attr in ["out_norm", "text_linear",
             "depformer_in", "depformer",
             "depformer_emb", "depformer_text_emb", "linears"]:
    if hasattr(teacher_lm, attr):
        getattr(teacher_lm, attr).to("cuda:1")
print("  Heads + Depformer → cuda:1")

mimi = mimi.to(device="cuda:0", dtype=torch.float16)
print("  Mimi → cuda:0 (fp16)")

torch.cuda.synchronize()
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"cuda:{i} after shard: free {free/1e9:.2f} / total {total/1e9:.2f} GB")

l0  = next(teacher_lm.transformer.layers[0].parameters()).device
l31 = next(teacher_lm.transformer.layers[31].parameters()).device
assert str(l0)  == "cuda:0", f"layer 0 on wrong device: {l0}"
assert str(l31) == "cuda:1", f"layer 31 on wrong device: {l31}"
print(f"\nTeacher layer  0 → {l0}")
print(f"Teacher layer 31 → {l31}")

_orig_layers = teacher_lm.transformer.layers
def _sharded_forward(x, *args, **kwargs):
    for layer in _orig_layers[:SPLIT]:
        x = layer(x)
    x = x.to("cuda:1", non_blocking=True)
    for layer in _orig_layers[SPLIT:]:
        x = layer(x)
    return x
teacher_lm.transformer.forward = _sharded_forward
print("Cross-GPU forward patch applied")

teacher_lm.eval()  # S1 is inference-only for the teacher
for p in teacher_lm.parameters():
    p.requires_grad_(False)
print("Teacher set to eval() and all params frozen")

print("\nWarming up teacher KV cache …")
with torch.inference_mode():
    dummy = torch.zeros(1, teacher_lm.num_codebooks, 8,
                        dtype=torch.long, device="cuda:0")
    _ = teacher_lm.forward_text(dummy)
print("Teacher warmup OK")
print("\n=== Cell 4 PASSED ===")


[attempt 1] model.safetensors  0.00 / ~14.00 GB


model.safetensors:   0%|          | 0.00/15.4G [00:00<?, ?B/s]

  OK
[attempt 1] tokenizer-e351c8d8-checkpoint125.safetensors  0.00 / ~0.35 GB


tokenizer-e351c8d8-checkpoint125.safeten(…):   0%|          | 0.00/385M [00:00<?, ?B/s]

  OK
[attempt 1] tokenizer_spm_32k_3.model  0.00 / ~0.00 GB


tokenizer_spm_32k_3.model:   0%|          | 0.00/553k [00:00<?, ?B/s]

  OK
  model.safetensors                                       15.376 GB
  tokenizer-e351c8d8-checkpoint125.safetensors            0.385 GB
  tokenizer_spm_32k_3.model                               0.001 GB

Loading teacher LM to CPU …
Loading Mimi to CPU …

Teacher LM params     : 7.688 B
Teacher LM mem (fp16) : 15.38 GB
Mimi params           : 79.3 M

Deleting weights from /tmp (loaded into RAM, disk copy not needed) …
Deleted /tmp/moshiko-weights
Disk free: 20.9 GB

Clearing GPU memory before shard …
  cuda:0 before shard: free 15.53 / total 15.64 GB
  cuda:1 before shard: free 15.53 / total 15.64 GB

Sharding teacher layer-by-layer …
  Layers  0-15 → cuda:0
  Layers 16-31 → cuda:1
  Heads + Depformer → cuda:1
  Mimi → cuda:0 (fp16)
cuda:0 after shard: free 8.21 / total 15.64 GB
cuda:1 after shard: free 7.13 / total 15.64 GB

Teacher layer  0 → cuda:0
Teacher layer 31 → cuda:1
Cross-GPU forward patch applied
Teacher set to eval() and all params frozen

Warming up teacher KV cache …


## Cell 5 — Stream pilot LibriLight slice to /tmp

**Target.** ~600 × 30 s windows ≈ 5 h audio.

**Source choice (priority order):**
1. Kaggle dataset with LibriLight if the user has pre-uploaded one (checked via
   `/kaggle/input/librilight-*` paths).
2. HF `datasets` streaming: `MLCommons/peoples_speech` is ~30 k h and public —
   used as a LibriLight-style substitute if LibriLight itself is gated.
3. `librispeech_asr` (train-clean-100) as a last-resort fallback.

All audio is resampled to Mimi's 24 kHz before caching. We truncate / skip
anything shorter than 30 s so every cache key has the same shape.


In [7]:
import pathlib, os, time
import numpy as np
import soundfile as sf
import soxr

PILOT_DIR = pathlib.Path("/tmp/pilot_audio")
PILOT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR          = 24_000
WINDOW_SECONDS     = 30
SAMPLES_PER_WINDOW = TARGET_SR * WINDOW_SECONDS   # 720 000
TARGET_WINDOWS     = 600                          # pilot budget (~5 h audio)

# ── Source: LibriSpeech train-clean-100 from attached Kaggle Dataset ─────────
# Attach the dataset in your notebook via Add Data → search 'librispeech'.
# Common slugs: 'librispeech', 'librispeech-clean', 'the-librispeech-dataset'.
# The cell tries several known paths and falls back to HF download if none found.

LIBRISPEECH_KAGGLE_CANDIDATES = [
    # Your exact dataset path (tasfiatanha/librispeech-train-clean-100)
    pathlib.Path("/kaggle/input/datasets/tasfiatanha/librispeech-train-clean-100/LibriSpeech/train-clean-100"),
    pathlib.Path("/kaggle/input/librispeech/train-clean-100"),
    pathlib.Path("/kaggle/input/librispeech-clean/train-clean-100"),
    pathlib.Path("/kaggle/input/the-librispeech-dataset/train-clean-100"),
    pathlib.Path("/kaggle/input/librispeech-asr/train-clean-100"),
    # Fallback: any /kaggle/input/ dir that contains .flac files
    *[p for p in pathlib.Path("/kaggle/input").glob("*")
      if p.is_dir() and list(p.rglob("*.flac"))[:1]],
]

librispeech_root = None
for candidate in LIBRISPEECH_KAGGLE_CANDIDATES:
    if candidate.exists() and list(candidate.rglob("*.flac"))[:1]:
        librispeech_root = candidate
        print(f"LibriSpeech found at: {librispeech_root}")
        break

if librispeech_root is None:
    print("LibriSpeech not found in /kaggle/input — downloading train-clean-100 from HF ...")
    from huggingface_hub import snapshot_download
    librispeech_root = pathlib.Path("/tmp/librispeech")
    librispeech_root.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id="openslr/librispeech_asr",
        repo_type="dataset",
        local_dir=str(librispeech_root),
        allow_patterns=["data/train-clean-100*"],
        local_dir_use_symlinks=False,
    )
    print("Download complete")

SOURCE = "librispeech_train_clean_100"
print(f"Audio source: {SOURCE}")

# ── Collect audio paths ───────────────────────────────────────────────────────
audio_paths = sorted(librispeech_root.rglob("*.flac"))
print(f"Found {len(audio_paths)} .flac files")
assert audio_paths, f"No .flac files found under {librispeech_root}"

# ── Splice into 30 s windows ──────────────────────────────────────────────────
windows     = []
transcripts = []
buf         = np.zeros(0, dtype=np.float32)
idx         = 0
t0          = time.time()

while len(windows) < TARGET_WINDOWS and idx < len(audio_paths):
    try:
        data, sr = sf.read(str(audio_paths[idx]), dtype="float32")
    except Exception as e:
        print(f"  skip {audio_paths[idx].name}: {e}")
        idx += 1
        continue
    if data.ndim > 1:
        data = data.mean(axis=1).astype(np.float32)
    if sr != TARGET_SR:
        data = soxr.resample(data, sr, TARGET_SR, quality="HQ").astype(np.float32)
    buf = np.concatenate([buf, data])
    while buf.shape[0] >= SAMPLES_PER_WINDOW and len(windows) < TARGET_WINDOWS:
        windows.append(buf[:SAMPLES_PER_WINDOW].copy())
        transcripts.append("")  # transcripts not needed for KD caching
        buf = buf[SAMPLES_PER_WINDOW:]
    idx += 1
    if idx % 100 == 0:
        print(f"  [{idx}/{len(audio_paths)}] {len(windows)} windows  "
              f"({time.time()-t0:.0f}s)")

print(f"\nCollected {len(windows)} windows x {WINDOW_SECONDS}s = "
      f"{len(windows)*WINDOW_SECONDS/3600:.2f} h audio")
assert len(windows) >= 100, (
    f"Only {len(windows)} windows collected — check dataset path")

# Save as mmap-able .npy for the cache loop
pilot_npy = PILOT_DIR / "windows.npy"
np.save(pilot_npy, np.stack(windows).astype(np.float32))
(PILOT_DIR / "transcripts.txt").write_text("\n".join(transcripts))
print(f"Saved {pilot_npy} ({pilot_npy.stat().st_size/1e9:.2f} GB)")
n_windows = len(windows)
del windows, buf
import gc; gc.collect()
print("=== Cell 5 PASSED ===")


LibriSpeech found at: /kaggle/input/datasets/tasfiatanha/librispeech-train-clean-100/LibriSpeech/train-clean-100
Audio source: librispeech_train_clean_100
Found 28539 .flac files
  [100/28539] 46 windows  (2s)
  [200/28539] 81 windows  (3s)
  [300/28539] 121 windows  (5s)
  [400/28539] 167 windows  (7s)
  [500/28539] 207 windows  (9s)
  [600/28539] 250 windows  (10s)
  [700/28539] 292 windows  (12s)
  [800/28539] 329 windows  (14s)
  [900/28539] 370 windows  (15s)
  [1000/28539] 403 windows  (17s)
  [1100/28539] 439 windows  (19s)
  [1200/28539] 480 windows  (20s)
  [1300/28539] 522 windows  (22s)
  [1400/28539] 560 windows  (24s)

Collected 600 windows x 30s = 5.00 h audio
Saved /tmp/pilot_audio/windows.npy (1.73 GB)
=== Cell 5 PASSED ===


## Cell 6 — LMDB writer and key schema

Each window is stored under a zero-padded decimal key (`b"000000"` …) so the
LMDB cursor order matches insertion order. Value layout is a dict pickled with
`pickle.HIGHEST_PROTOCOL`:

```python
{
    "hidden": np.ndarray[375, 4096] fp16,         # teacher h_T_final
    "text_top_idx": np.ndarray[375, 256] int32,   # top-256 token ids
    "text_top_val": np.ndarray[375, 256] fp16,    # top-256 raw logits
    "semantic": np.ndarray[375, 2048] fp16,       # cb0 (semantic) logits
    "source": str,                                # provenance tag
}
```

`text_top_*` is a sparse approximation of the full 32 000-wide distribution;
storing the full logits for 60 k × 375 positions blows past Kaggle Dataset
quotas (would be ~35 GB for 60 k windows even in fp16). For KD this is plenty
— top-256 covers the tail well past Moshi's own inner-monologue entropy.


In [8]:
import lmdb, pickle, pathlib, struct

CACHE_DIR = pathlib.Path("/kaggle/working/teacher_cache_pilot")
CACHE_DIR.mkdir(parents=True, exist_ok=True)
LMDB_PATH = CACHE_DIR / "pilot.lmdb"

# 30 GB envelope is generous — actual usage will be well under 10 GB.
# LMDB requires map_size be declared up front; resizing requires a reopen.
MAP_SIZE = 30 * 1024 ** 3

# Delete any stale env so a partial run doesn't corrupt new keys.
if LMDB_PATH.exists():
    print(f"Removing stale {LMDB_PATH}")
    import shutil; shutil.rmtree(LMDB_PATH)

env = lmdb.open(
    str(LMDB_PATH),
    map_size     = MAP_SIZE,
    subdir       = True,
    lock         = True,
    sync         = True,      # flush on every commit — safe against session kill
    writemap     = True,
    map_async    = True,
    readahead    = False,
    max_readers  = 4,
)

def cache_key(i: int) -> bytes:
    return f"{i:06d}".encode("ascii")

def write_window(i: int, payload: dict):
    with env.begin(write=True) as txn:
        txn.put(cache_key(i), pickle.dumps(payload, protocol=pickle.HIGHEST_PROTOCOL))

def read_back(i: int):
    with env.begin() as txn:
        raw = txn.get(cache_key(i))
    return pickle.loads(raw) if raw is not None else None

# Smoke test — write one dummy key and read it back.
import numpy as np
dummy = {
    "hidden"       : np.zeros((2, 4), dtype=np.float16),
    "text_top_idx" : np.zeros((2, 4), dtype=np.int32),
    "text_top_val" : np.zeros((2, 4), dtype=np.float16),
    "semantic"     : np.zeros((2, 4), dtype=np.float16),
    "source"       : "smoke",
}
write_window(999_999, dummy)
rt = read_back(999_999)
assert rt is not None and rt["source"] == "smoke"
with env.begin(write=True) as txn:
    txn.delete(cache_key(999_999))
print(f"LMDB ready at {LMDB_PATH} (map_size={MAP_SIZE/1e9:.0f} GB)")
print("=== Cell 6 PASSED ===")


LMDB ready at /kaggle/working/teacher_cache_pilot/pilot.lmdb (map_size=32 GB)
=== Cell 6 PASSED ===


## Cell 7 — Pilot caching loop

Per window:
1. `mimi.encode(wav)` → int64 codes `[1, 8, 375]` (8 audio codebooks × 375 frames @ 12.5 Hz).
2. Prepend a text-silence codebook (token id 3 = moshi's text pad) to get a
   17-codebook tensor `[1, 17, 375]`, matching what `forward_text` expects.
3. `teacher_lm.forward_text(codes)` in `inference_mode`.
4. Extract hidden (pre-`text_linear`), full text logits, full semantic logits.
5. Top-256 truncation on text logits.
6. Pickle + LMDB put.

We fold windows into mini-batches of 2 to amortise kernel launches. At B=2,
peak memory during teacher forward is ~12 GB on cuda:1 — leaves room for
kv-cache growth across the 375-step window.

Throughput is logged every 25 windows so a crash still gives us extrapolation
data.


In [9]:
import time, json, pathlib
import numpy as np
import torch

THROUGHPUT_LOG = pathlib.Path("/kaggle/working/S1_throughput.json")
pilot_npy      = pathlib.Path("/tmp/pilot_audio/windows.npy")

# Mmap the audio — avoids loading the full 13 GB into RAM.
audio_all = np.load(pilot_npy, mmap_mode="r")
n_windows = audio_all.shape[0]
print(f"Starting cache loop over {n_windows} windows")

# Moshi conventions:
#   num_codebooks = 17 (1 text + 16 audio).
#   forward_text consumes 17 codebooks but only uses codebooks 0..8 semantically
#   (text + cb0 semantic + cb1..cb7 acoustic). We encode with Mimi's 8 audio
#   codebooks, zero-pad the rest, and use a pad token for text.
NUM_CB_TEACHER = teacher_lm.num_codebooks
assert NUM_CB_TEACHER == 17, NUM_CB_TEACHER

TEXT_PAD = 3   # moshi's text pad id — see moshi.models.loaders
MIMI_AUDIO_CB = 8

# Pre-allocate a reusable codes buffer.
T_FRAMES = 375
batch_codes = torch.full(
    (1, NUM_CB_TEACHER, T_FRAMES),
    TEXT_PAD, dtype=torch.long, device="cuda:0",
)

TOPK = 256
log_rows = []
n_done = 0
t_loop_start = time.time()

for i in range(n_windows):
    t_w_start = time.time()

    # 1. Load window, move to GPU.
    wav = torch.from_numpy(audio_all[i].copy()).to(
        device="cuda:0", dtype=torch.float16
    ).unsqueeze(0).unsqueeze(0)  # [1, 1, 720000]

    try:
        with torch.inference_mode():
            # 2. Mimi encode → [1, 8, 375] int64.
            codes = mimi.encode(wav)
            assert codes.shape[-1] == T_FRAMES, (
                f"unexpected Mimi frame count {codes.shape[-1]}")
            assert codes.shape[1] == MIMI_AUDIO_CB, codes.shape

            # 3. Assemble 17-codebook tensor. cb0 = text(pad), cb1..8 = audio.
            batch_codes.fill_(TEXT_PAD)
            batch_codes[:, 1:1 + MIMI_AUDIO_CB, :] = codes.to(torch.long)

            # 4. Teacher forward.
            hidden, text_logits = teacher_lm.forward_text(batch_codes)
            # hidden:        [1, T, 4096]          (on cuda:1)
            # text_logits:   [1, 1, T, vocab]      (on cuda:1)

            # 5. Pull semantic logits from the Depformer-linked head.
            # teacher_lm.linears is a ModuleList mapping hidden → cb-logits per codebook.
            # Semantic logits omitted: Depformer is a streaming module
            # (processes S=1 at a time) and cannot run over 375 frames
            # in a single forward call. Phase-1 KD only needs hidden +
            # text logits; semantic logits are a Phase-2+ addition.
            semantic_logits = None
            # semantic_logits: [1, T, 2048]

            # 6. Top-K on text logits.
            text_logits = text_logits.squeeze(0).squeeze(0)  # [T, vocab]
            topk_val, topk_idx = text_logits.topk(TOPK, dim=-1)

        payload = {
            "hidden":       hidden.squeeze(0).to("cpu", dtype=torch.float16).numpy(),
            "text_top_idx": topk_idx.to("cpu", dtype=torch.int32).numpy(),
            "text_top_val": topk_val.to("cpu", dtype=torch.float16).numpy(),
            "semantic":     semantic_logits.squeeze(0).to("cpu", dtype=torch.float16).numpy()
                           if semantic_logits is not None else None,
            "source":       SOURCE,
        }
        write_window(i, payload)
        n_done += 1

        # Periodic sync every 50 windows — safe against session kill
        if n_done % 50 == 0:
            env.sync()

    except RuntimeError as e:
        # Don't let a single bad window kill the session.
        print(f"  window {i} FAILED: {type(e).__name__}: {e}")
        torch.cuda.empty_cache()
        continue

    t_w = time.time() - t_w_start
    if (i + 1) % 25 == 0 or i == n_windows - 1:
        elapsed = time.time() - t_loop_start
        wpm = n_done / max(elapsed, 1e-6) * 60
        row = {
            "window_idx": i,
            "n_done":     n_done,
            "elapsed_s":  round(elapsed, 1),
            "last_w_ms":  round(t_w * 1000, 1),
            "wpm":        round(wpm, 2),
            "gpu0_free":  torch.cuda.mem_get_info(0)[0] / 1e9,
            "gpu1_free":  torch.cuda.mem_get_info(1)[0] / 1e9,
        }
        log_rows.append(row)
        print(f"  [{i+1:4d}/{n_windows}]  wpm={wpm:5.1f}  "
              f"last={t_w*1000:6.0f}ms  gpu0_free={row['gpu0_free']:.1f}  "
              f"gpu1_free={row['gpu1_free']:.1f}")

env.sync()
elapsed = time.time() - t_loop_start
final = {
    "n_windows_written": n_done,
    "wall_seconds":      round(elapsed, 1),
    "windows_per_minute": round(n_done / elapsed * 60, 2) if elapsed > 0 else 0,
    "source":            SOURCE,
    "topk":              TOPK,
    "t_frames":          T_FRAMES,
    "log":               log_rows,
}
THROUGHPUT_LOG.write_text(json.dumps(final, indent=2))
print(f"\nCache complete: {n_done}/{n_windows} windows, "
      f"{final['windows_per_minute']} w/min")
print(f"Throughput log → {THROUGHPUT_LOG}")

# Sanity: read back last window and check shapes.
rb = read_back(n_done - 1)
assert rb is not None
assert rb["hidden"].shape       == (T_FRAMES, 4096), rb["hidden"].shape
assert rb["text_top_idx"].shape == (T_FRAMES, TOPK), rb["text_top_idx"].shape
assert rb["text_top_val"].shape == (T_FRAMES, TOPK), rb["text_top_val"].shape
# semantic may be None in Phase-0 cache (Depformer skipped)
if rb["semantic"] is not None:
    assert rb["semantic"].shape == (T_FRAMES, 2048), rb["semantic"].shape
print("Read-back shapes OK")
print("=== Cell 7 PASSED ===")


Starting cache loop over 600 windows
  [  25/600]  wpm=189.6  last=   289ms  gpu0_free=7.5  gpu1_free=7.0
  [  50/600]  wpm=197.4  last=   295ms  gpu0_free=7.5  gpu1_free=7.0
  [  75/600]  wpm=199.6  last=   299ms  gpu0_free=7.5  gpu1_free=7.0
  [ 100/600]  wpm=199.8  last=   308ms  gpu0_free=7.5  gpu1_free=7.0
  [ 125/600]  wpm=199.6  last=   304ms  gpu0_free=7.5  gpu1_free=7.0
  [ 150/600]  wpm=198.4  last=   308ms  gpu0_free=7.5  gpu1_free=7.0
  [ 175/600]  wpm=198.0  last=   309ms  gpu0_free=7.5  gpu1_free=7.0
  [ 200/600]  wpm=197.4  last=   310ms  gpu0_free=7.5  gpu1_free=7.0
  [ 225/600]  wpm=196.7  last=   319ms  gpu0_free=7.5  gpu1_free=7.0
  [ 250/600]  wpm=195.4  last=   324ms  gpu0_free=7.5  gpu1_free=7.0
  [ 275/600]  wpm=194.7  last=   320ms  gpu0_free=7.5  gpu1_free=7.0
  [ 300/600]  wpm=194.0  last=   323ms  gpu0_free=7.5  gpu1_free=7.0
  [ 325/600]  wpm=193.4  last=   317ms  gpu0_free=7.5  gpu1_free=7.0
  [ 350/600]  wpm=193.0  last=   318ms  gpu0_free=7.5  gpu1_free=7

## Cell 8 — Extract frozen-heads state_dicts

These are modules Phase 1 will load verbatim into the student shell built in
S0 (which used `load_weight=False`). We copy off GPU → CPU fp16 because the
dataset will be pulled into Kaggle notebooks next session; fp16 halves the
download.


In [ ]:
import torch, pathlib, json

FROZEN_DIR = pathlib.Path("/kaggle/working/moshi_frozen_heads")
FROZEN_DIR.mkdir(parents=True, exist_ok=True)

FROZEN_MODULES = [
    "emb",                 # audio-stream embeddings (17 streams → 4096)
    "text_emb",            # text embedding
    "out_norm",            # pre-head norm
    "text_linear",         # text head (→ 32 000)
    "depformer_in",        # Depformer input adapter
    "depformer",           # Depformer blocks
    "depformer_emb",       # Depformer positional embeddings
    "depformer_text_emb",  # Depformer text embeddings
    "linears",             # per-codebook output linears
]

def to_cpu_fp16(sd):
    # S0 frozen heads load in student shell as fp16 too.
    out = {}
    for k, v in sd.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.detach().to("cpu", dtype=torch.float16) \
                     if v.is_floating_point() else v.detach().to("cpu")
        else:
            out[k] = v
    return out

manifest = {}
for name in FROZEN_MODULES:
    if not hasattr(teacher_lm, name):
        print(f"  WARNING: teacher has no attribute `{name}` — skipping")
        continue
    module = getattr(teacher_lm, name)
    sd = to_cpu_fp16(module.state_dict())
    out_path = FROZEN_DIR / f"{name}.pt"
    torch.save(sd, out_path)
    size_mb = out_path.stat().st_size / 1e6
    n_params = sum(v.numel() for v in sd.values() if isinstance(v, torch.Tensor))
    manifest[name] = {
        "path_relative": out_path.name,
        "size_mb":       round(size_mb, 2),
        "n_params":      int(n_params),
        "n_tensors":     len(sd),
        "example_keys":  list(sd.keys())[:3],
    }
    print(f"  {name:24s} {size_mb:7.1f} MB  {n_params/1e6:6.1f} M params")

(FROZEN_DIR / "frozen_heads_manifest.json").write_text(json.dumps(manifest, indent=2))

total_mb = sum(m["size_mb"] for m in manifest.values())
total_params = sum(m["n_params"] for m in manifest.values())
print(f"\nTotal frozen-head size: {total_mb/1024:.2f} GB")
print(f"Total frozen-head params: {total_params/1e6:.1f} M")

# Required-module assertion — Phase 1 will break without any of these.
required = {"emb", "text_emb", "out_norm", "text_linear", "depformer", "linears"}
missing = required - set(manifest.keys())
assert not missing, f"Missing required frozen modules: {missing}"
print("\nAll required frozen-head modules present")
print("=== Cell 8 PASSED ===")


## Cell 9 — MANIFEST.md, env.txt, cache size audit

Two output directories (`teacher_cache_pilot/`, `moshi_frozen_heads/`) become
two separate Kaggle datasets. Summarise both so the next session can
sanity-check without re-running.


In [ ]:
import os, json, subprocess, time, pathlib
import torch, transformers

WORKING        = pathlib.Path("/kaggle/working")
CACHE_DIR      = WORKING / "teacher_cache_pilot"
FROZEN_DIR     = WORKING / "moshi_frozen_heads"
THROUGHPUT_LOG = WORKING / "S1_throughput.json"

throughput = json.loads(THROUGHPUT_LOG.read_text())
frozen_manifest = json.loads((FROZEN_DIR / "frozen_heads_manifest.json").read_text())

def dir_size_gb(p: pathlib.Path) -> float:
    return sum(f.stat().st_size for f in p.rglob("*") if f.is_file()) / 1e9

cache_gb  = dir_size_gb(CACHE_DIR)
frozen_gb = dir_size_gb(FROZEN_DIR)

# ── Extrapolation: how many S2..Sk sessions to reach 60 k windows? ────────────
# A Kaggle session caps at ~9 h; assume ~8 h of usable runtime after setup.
wpm = throughput["windows_per_minute"] or 1.0
target_windows = 60_000
session_capacity = wpm * 60 * 8  # windows per 8 h session
needed_sessions = max(1, round(target_windows / max(session_capacity, 1)))
projected_cache_gb = cache_gb * target_windows / max(throughput["n_windows_written"], 1)

# ── env.txt ──
env_out  = subprocess.run(["pip", "freeze"],    capture_output=True, text=True).stdout
nvid_out = subprocess.run(["nvidia-smi", "-q"], capture_output=True, text=True).stdout
(CACHE_DIR  / "env.txt").write_text(env_out + "\n=== nvidia-smi ===\n" + nvid_out)
(FROZEN_DIR / "env.txt").write_text(env_out + "\n=== nvidia-smi ===\n" + nvid_out)

# ── MANIFEST.md for cache dataset ──
(CACHE_DIR / "MANIFEST.md").write_text(f"""# MANIFEST — moshi-teacher-cache-pilot

Phase-0 teacher targets, pilot slice.

| Key | Value |
|---|---|
| source | `{throughput['source']}` |
| n_windows | {throughput['n_windows_written']} |
| window_seconds | 30 |
| hours_of_audio | {throughput['n_windows_written'] * 30 / 3600:.2f} |
| topk_text | {throughput['topk']} |
| t_frames | {throughput['t_frames']} |
| hidden_dim | 4096 (teacher fp16) |
| semantic_dim | 2048 (cb0 fp16) |
| windows_per_minute | {throughput['windows_per_minute']} |
| cache_size_gb | {cache_gb:.2f} |
| torch | {torch.__version__} |
| transformers | {transformers.__version__} |

## LMDB key schema
Keys are 6-digit zero-padded ASCII decimals (`b"000000"`, `b"000001"`, …).
Values are pickled dicts:
```
hidden       : float16  [375, 4096]
text_top_idx : int32    [375, 256]
text_top_val : float16  [375, 256]
semantic     : float16  [375, 2048]
source       : str
```

## Extrapolation for full Phase 0
At {throughput['windows_per_minute']} w/min, 8 h of usable runtime per session,
the 60 000-window target needs **≈{needed_sessions} sessions** and
**≈{projected_cache_gb:.0f} GB** of LMDB.
""")

# ── MANIFEST.md for frozen-heads dataset ──
modules_tbl = "\n".join(
    f"| `{name}` | {m['path_relative']} | {m['n_params']/1e6:.1f} M | {m['size_mb']:.1f} MB |"
    for name, m in frozen_manifest.items()
)
(FROZEN_DIR / "MANIFEST.md").write_text(f"""# MANIFEST — moshi-frozen-heads

Teacher state_dicts that the student shell loads verbatim (student was built
with `load_weight=False` in S0).

| Module | File | Params | Size |
|---|---|---|---|
{modules_tbl}

Total: {sum(m['n_params'] for m in frozen_manifest.values())/1e6:.1f} M params,
{sum(m['size_mb'] for m in frozen_manifest.values())/1024:.2f} GB fp16.

Load pattern (next session):
```python
for name in FROZEN_MODULES:
    sd = torch.load(f"/kaggle/input/moshi-frozen-heads/{{name}}.pt", map_location="cpu")
    getattr(student_lm, name).load_state_dict(sd)
```
""")

print(f"Cache dir  : {CACHE_DIR}  {cache_gb:.2f} GB")
print(f"Frozen dir : {FROZEN_DIR} {frozen_gb:.2f} GB")
print(f"windows_per_minute = {throughput['windows_per_minute']}")
print(f"Projected full Phase 0: ~{needed_sessions} sessions, "
      f"~{projected_cache_gb:.0f} GB cache")

stat = os.statvfs("/kaggle/working")
print(f"Disk free: {stat.f_bavail * stat.f_frsize / 1e9:.1f} GB")
print("=== Cell 9 PASSED ===")


In [10]:
print("This cell is running")
import lmdb, pathlib, shutil

CACHE_DIR    = pathlib.Path("/kaggle/working/teacher_cache_pilot")
LMDB_PATH    = CACHE_DIR / "pilot.lmdb"
COMPACT_PATH = CACHE_DIR / "pilot_compact.lmdb"

# Force close any open handle in this process
try:
    env.close()
    print("Closed existing env handle")
except:
    pass

# Also try closing via a fresh open (lmdb tracks open envs per process)
try:
    _tmp = lmdb.open(str(LMDB_PATH), readonly=True, lock=False)
    _tmp.close()
except:
    pass

if COMPACT_PATH.exists():
    shutil.rmtree(COMPACT_PATH)

src_env = lmdb.open(str(LMDB_PATH), readonly=True, lock=False, max_readers=1)
dst_env = lmdb.open(str(COMPACT_PATH), map_size=4 * 1024**3)

with src_env.begin() as src_txn:
    with dst_env.begin(write=True) as dst_txn:
        n = 0
        for key, val in src_txn.cursor():
            dst_txn.put(key, val)
            n += 1

src_env.close()
dst_env.sync()
dst_env.close()

print(f"Copied {n} entries")
print(f"Original : {(LMDB_PATH/'data.mdb').stat().st_size/1e9:.1f} GB")
print(f"Compacted: {(COMPACT_PATH/'data.mdb').stat().st_size/1e9:.1f} GB")

shutil.rmtree(LMDB_PATH)
COMPACT_PATH.rename(LMDB_PATH)
print("Done — pilot.lmdb compacted")

This cell is running
Closed existing env handle
Copied 600 entries
Original : 32.2 GB
Compacted: 2.2 GB
Done — pilot.lmdb compacted


## Cell 10 — Push `moshi-teacher-cache-pilot`

The LMDB env must be closed before pushing, otherwise the uploader sees
half-written pages.


In [12]:
import subprocess, json, pathlib, os

# Close LMDB before upload.
try:
    env.sync()
    env.close()
    print("LMDB env closed")
except NameError:
    print("env already closed (kernel restart?)")

CACHE_DIR = pathlib.Path("/kaggle/working/teacher_cache_pilot")
username  = os.environ.get("KAGGLE_USERNAME", "tasfiatanha")

metadata = {
    "title":    "moshi-teacher-cache-pilot",
    "id":       f"{username}/moshi-teacher-cache-pilot",
    "licenses": [{"name": "CC0-1.0"}],
}
(CACHE_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
print(f"Dataset id: {username}/moshi-teacher-cache-pilot")

print("\nFiles to push:")
for p in sorted(CACHE_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(CACHE_DIR)
        print(f"  {str(rel):<50s} {p.stat().st_size/1e6:8.1f} MB")

print("\nRunning: kaggle datasets create …")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(CACHE_DIR), "--dir-mode", "zip"],
    capture_output=True, text=True,
)
print(r.stdout or "(no stdout)")

if r.returncode == 0:
    print(f"SUCCESS — kaggle.com/{username}/moshi-teacher-cache-pilot")
else:
    print(f"Create failed (rc={r.returncode}), trying version bump …")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version",
         "-p", str(CACHE_DIR),
         "-m", "S1 pilot cache","--dir-mode", "zip"],
        capture_output=True, text=True,
    )
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0:
        print("STDERR:", r2.stderr)
        print("\nManual fallback:")
        print(f"  cd {CACHE_DIR} && kaggle datasets create -p .")
    else:
        print(f"SUCCESS — kaggle.com/{username}/moshi-teacher-cache-pilot")


Error: Attempt to operate on closed/deleted/dropped object.

In [13]:
import subprocess, json, pathlib, os

CACHE_DIR = pathlib.Path("/kaggle/working/teacher_cache_pilot")
username  = os.environ.get("KAGGLE_USERNAME", "tasfiatanha")

metadata = {
    "title":    "moshi-teacher-cache-pilot",
    "id":       f"{username}/moshi-teacher-cache-pilot",
    "licenses": [{"name": "CC0-1.0"}],
}
(CACHE_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

print("Files to push:")
for p in sorted(CACHE_DIR.rglob("*")):
    if p.is_file():
        print(f"  {str(p.relative_to(CACHE_DIR)):<50s} {p.stat().st_size/1e6:8.1f} MB")

# Dataset already exists — version bump directly
print("\nRunning: kaggle datasets version ...")
r = subprocess.run(
    ["kaggle", "datasets", "version",
     "-p", str(CACHE_DIR),
     "-m", "S1 pilot cache compacted 600 windows",
     "--dir-mode", "zip"],
    capture_output=True, text=True,
)
print(r.stdout or "(no stdout)")
if r.returncode != 0:
    print("STDERR:", r.stderr)
else:
    print(f"SUCCESS — kaggle.com/{username}/moshi-teacher-cache-pilot")

Files to push:
  dataset-metadata.json                                   0.0 MB
  pilot.lmdb/data.mdb                                  2189.8 MB
  pilot.lmdb/lock.mdb                                     0.0 MB

Running: kaggle datasets version ...
Starting upload for file pilot.lmdb.zip
Upload successful: pilot.lmdb.zip (2GB)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/tasfiatanha/moshi-teacher-cache-pilot

SUCCESS — kaggle.com/tasfiatanha/moshi-teacher-cache-pilot


## Cell 11 — Push `moshi-frozen-heads`


In [ ]:
import subprocess, json, pathlib, os

FROZEN_DIR = pathlib.Path("/kaggle/working/moshi_frozen_heads")
username   = os.environ.get("KAGGLE_USERNAME", "tasfiatanha")

metadata = {
    "title":    "moshi-frozen-heads",
    "id":       f"{username}/moshi-frozen-heads",
    "licenses": [{"name": "CC0-1.0"}],
}
(FROZEN_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))
print(f"Dataset id: {username}/moshi-frozen-heads")

print("\nFiles to push:")
for p in sorted(FROZEN_DIR.rglob("*")):
    if p.is_file():
        rel = p.relative_to(FROZEN_DIR)
        print(f"  {str(rel):<50s} {p.stat().st_size/1e6:8.1f} MB")

print("\nRunning: kaggle datasets create …")
r = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(FROZEN_DIR)],
    capture_output=True, text=True,
)
print(r.stdout or "(no stdout)")

if r.returncode == 0:
    print(f"SUCCESS — kaggle.com/{username}/moshi-frozen-heads")
else:
    print(f"Create failed (rc={r.returncode}), trying version bump …")
    r2 = subprocess.run(
        ["kaggle", "datasets", "version",
         "-p", str(FROZEN_DIR),
         "-m", "S1 frozen-heads export"],
        capture_output=True, text=True,
    )
    print(r2.stdout or "(no stdout)")
    if r2.returncode != 0:
        print("STDERR:", r2.stderr)
        print("\nManual fallback:")
        print(f"  cd {FROZEN_DIR} && kaggle datasets create -p .")
    else:
        print(f"SUCCESS — kaggle.com/{username}/moshi-frozen-heads")

print("\n=== S1 COMPLETE ===")
print("Next: update MoshiCompressionProgress.md and plan S2 (full Phase 0 cache).")


In [11]:
  !ls -la /kaggle/working/teacher_cache_pilot/pilot.lmdb/
  !du -h --apparent-size /kaggle/working/teacher_cache_pilot/pilot.lmdb/data.mdb
  !du -h /kaggle/working/teacher_cache_pilot/pilot.lmdb/data.mdb

total 2138444
drwxr-xr-x 2 root root       4096 Apr 13 20:04 .
drwxr-xr-x 3 root root       4096 Apr 13 20:05 ..
-rw-r--r-- 1 root root 2189750272 Apr 13 20:05 data.mdb
-rw-r--r-- 1 root root       8192 Apr 13 20:04 lock.mdb
2.1G	/kaggle/working/teacher_cache_pilot/pilot.lmdb/data.mdb
2.1G	/kaggle/working/teacher_cache_pilot/pilot.lmdb/data.mdb
